# 01. LangChain tool call


> 랭체인에서도 tool 사용 가능

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")

# llm.invoke([HumanMessage("하이~")])

In [2]:
from langchain_core.tools import tool

In [3]:
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록 / 이 함수에다가 invoke 하기 위해 필요
def get_current_time(timezone: str, location: str) -> str:
    ### 단순한 주석이 아니라 랭체인에 이 함수의 기능과 입력값, 사용 방법을 알려주는 문서화 문자열
    """ 현재 시각을 반환하는 함수 

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time


In [4]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

In [5]:
# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [6]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("서울은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

In [7]:
# messages

In [ ]:
# response.tool_calls

In [ ]:
# tool_call = response.tool_calls[0]
# tool_call["name"]

In [ ]:
# tool_dict[tool_call["name"]]

In [ ]:
# tool_dict[tool_call["name"]].invoke(tool_call)

In [8]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

# messages  # 탐색용


{'timezone': 'Asia/Seoul', 'location': '서울'}
Asia/Seoul (서울) 현재시각 2026-06-26 16:23:43 


In [ ]:
# llm_with_tools.invoke(messages)

In [9]:
messages.append(llm_with_tools.invoke(messages))

In [ ]:
# messages

### 주가 예제

In [10]:
import yfinance as yf

@tool
def get_yf_stock_history(ticker: str, period: str) -> str:
    """ 
    주식 종목의 가격 데이터를 조회하는 함수

    Args:
        ticker (str): 주식 종목 코드 (예: AAPL)
        period (str): 주식 데이터 조회 기간 (예: 1d, 1mo, 1y)
    
    """
    stock = yf.Ticker(ticker)
    history = stock.history(period=period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [11]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
messages.append(response)

In [ ]:
# response.tool_calls

#### 이어서 자연어로 모델이 답변하는 것 추가 실습

In [12]:
for tool_call in response.tool_calls:
    out = tool_dict[tool_call['name']].invoke(tool_call)
    messages.append(out)

llm_with_tools.invoke(messages)


AIMessage(content='현재 테슬라(TSLA)의 주가는 한 달 전과 비교하여 아래와 같습니다:\n\n- **한 달 전 주가 (2026년 5월 26일)**: 433.59 USD\n- **현재 주가 (2026년 6월 25일)**: 375.12 USD\n\n결론적으로, 테슬라의 주가는 한 달 전에 비해 **하락**했습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 1673, 'total_tokens': 1763, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fff1d3b4b6', 'id': 'chatcmpl-DuvCnK9P4IvL9lSl1ycZLLcVL2nBD', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f02d0-5d6f-72f3-9a44-47d4ba694fd4-0', usage_metadata={'input_tokens': 1673, 'output_tokens': 90, 'total_tokens': 1763, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## 실습 1. pdf_summary.py

pdf_summary 하는 함수 tool로 등록하여 사용해보기

+ summarize_text 도 langchain 사용하는 것으로 변경

**주의:** PDF 셀은 위 테슬라 예제와 `messages`/`response` 변수를 공유합니다.  
`invoke` 셀(28번)을 **두 번 연속 실행**하면 400 에러가 납니다. 에러 나면 28번 셀부터 다시 실행하세요.

In [23]:
from langchain_core.messages import HumanMessage
import os 

pdf_path = os.path.join(os.getcwd(),"samples/videoanomaly.pdf")

messages = [
    HumanMessage(f"이 {pdf_path} 문서의 저자는 누구야?"),
]


In [24]:
from pdf_summary import summarize_pdf


In [25]:
tools = [summarize_pdf]
tool_dict = {"summarize_pdf": summarize_pdf}


In [ ]:
# llm은 맨 위 셀(1번)에서 이미 만듦 — 여기서 다시 만들 필요 없음
# llm = ChatOpenAI(model="gpt-4o-mini")


In [30]:
llm_with_tools = llm.bind_tools(tools)  # summarize_pdf만 바인딩


In [31]:
# 이 셀만 다시 실행해도 됨 — invoke 직전에 messages를 여기서 다시 만듦
messages = [HumanMessage(f"이 {pdf_path} 문서의 저자는 누구야?")]

response = llm_with_tools.invoke(messages)
messages.append(response)
response.tool_calls  # 'summarize_pdf' 여야 함


BadRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_K4HMll2cVrw3s8m8TVzHH6WF", 'type': 'invalid_request_error', 'param': 'messages', 'code': None}}

In [ ]:
for tool_call in response.tool_calls:
    out = tool_dict[tool_call['name']].invoke(tool_call)
    messages.append(out)


In [ ]:
final_response = llm_with_tools.invoke(messages)
messages.append(final_response)
final_response.content


In [ ]:
# final_response.content 셀에서 답변 확인
